# CELDA 1 — Carga y verificación del pipeline guardado

Cargamos el pipeline guardado y verificamos que está listo para producción.

In [ ]:
# =============================================================================
# CELDA 1 — Carga y verificación del pipeline guardado
# =============================================================================

# Importaciones principales
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set(font_scale=1.2)
warnings.filterwarnings('ignore')

# Añadir directorio src al path
sys.path.append(os.path.join('..', 'src'))

# Cargar pipeline
ruta_modelo = os.path.join('..', 'models', 'modelo_final.pkl')
pipeline = joblib.load(ruta_modelo)

print("Pipeline cargado exitosamente.")
print("\nComponentes del pipeline:")
for nombre, paso in pipeline.named_steps.items():
    print(f"  {nombre}: {type(paso).__name__}")

# Verificar que el modelo está listo para producción
print("\nVerificación del pipeline:")
print(f"  Número de pasos: {len(pipeline.steps)}")
print(f"  Modelo final: {type(pipeline.named_steps['modelo']).__name__}")
print("\n✓ Pipeline listo para producción.")

# CELDA 2 — Prueba del pipeline con datos nuevos de ejemplo

Construimos manualmente vectores de features de ejemplo (uno por sector)
y ejecutamos el pipeline para verificar que funciona correctamente.

In [ ]:
# =============================================================================
# CELDA 2 — Prueba del pipeline con datos nuevos de ejemplo
# =============================================================================

# Cargar dataset para obtener nombres de features
ruta_datos = os.path.join('..', 'data', 'processed', 'dataset_modelamiento.csv')
df = pd.read_csv(ruta_datos, index_col=0)

# Obtener nombres de features (excluyendo targets y sectores)
columnas_excluir = [col for col in df.columns if col.startswith('target_')] + \
                   [col for col in df.columns if col.endswith('_sector')]
nombres_features = [col for col in df.columns if col not in columnas_excluir]

print(f"Número de features: {len(nombres_features)}")
print(f"Features: {nombres_features}")

# Construir vectores de ejemplo por sector
# El modelo fue entrenado con 11 features (retornos base):
# SP500, VIX, BRENT, WTI, BOVESPA, MERVAL, USD_COP, GOLD, COPPER, EXXON, CHEVRON
# Usamos los valores de mediana y luego ajustamos por sector

ejemplos = {
    'energía': {
        # Sector energía: BRENT y WTI son los activos principales
        'BRENT': 0.025,   # Retorno positivo reciente
        'WTI': 0.020,
        'EXXON': 0.015,
        'CHEVRON': 0.018,
        'VIX': -0.010,   # VIX negativo = menor incertidumbre
        'SP500': 0.005,
        'BOVESPA': 0.003,
        'MERVAL': 0.002,
        'USD_COP': -0.005,
        'GOLD': 0.010,
        'COPPER': 0.008
    },
    'índice': {
        # Sector índice: SP500 y BOVESPA
        'SP500': 0.010,
        'BOVESPA': 0.015,
        'MERVAL': 0.012,
        'VIX': 0.005,
        'BRENT': 0.008,
        'WTI': 0.007,
        'USD_COP': -0.003,
        'GOLD': 0.005,
        'COPPER': 0.004,
        'EXXON': 0.006,
        'CHEVRON': 0.006
    },
    'divisa': {
        # Sector divisa: USD_COP
        'USD_COP': 0.030,   # Depreciación del peso (retorno positivo para USD/COP)
        'VIX': 0.015,     # Mayor incertidumbre
        'BRENT': 0.010,
        'WTI': 0.009,
        'SP500': 0.002,
        'BOVESPA': 0.001,
        'MERVAL': 0.001,
        'GOLD': 0.003,
        'COPPER': 0.002,
        'EXXON': 0.004,
        'CHEVRON': 0.004
    },
    'metal': {
        # Sector metal: GOLD y COPPER
        'GOLD': 0.018,    # Activos refugio
        'COPPER': 0.012,
        'VIX': 0.008,
        'BRENT': 0.006,
        'WTI': 0.005,
        'SP500': 0.003,
        'BOVESPA': 0.002,
        'MERVAL': 0.002,
        'USD_COP': -0.002,
        'EXXON': 0.004,
        'CHEVRON': 0.004
    },
    'volatilidad': {
        # Sector volatilidad: VIX
        'VIX': 0.045,     # Alto miedo en el mercado
        'BRENT': -0.010,
        'WTI': -0.012,
        'SP500': -0.015,   # Caída en índices
        'BOVESPA': -0.020,
        'MERVAL': -0.018,
        'USD_COP': 0.025,   # Fuga a dólares
        'GOLD': 0.020,     # Refugio
        'COPPER': -0.008,
        'EXXON': -0.009,
        'CHEVRON': -0.009
    }
}

# Crear DataFrame con ejemplos
df_ejemplos = pd.DataFrame(index=ejemplos.keys(), columns=nombres_features)

# Llenar con valores por defecto (mediana del dataset)
for col in nombres_features:
    df_ejemplos[col] = df[col].median()

print("\nValores de mediana usados:")
print(df_ejemplos.iloc[0].to_dict())

# Actualizar con valores específicos de cada ejemplo
for sector, valores in ejemplos.items():
    for feature, valor in valores.items():
        if feature in df_ejemplos.columns:
            df_ejemplos.loc[sector, feature] = valor
        else:
            print(f"ADVERTENCIA: {feature} no encontrado en columnas del DataFrame")

print("\nDataFrame de ejemplos (todas las columnas):")
print(df_ejemplos)

# Cargar scaler del entrenamiento
import joblib
import os
ruta_scaler = os.path.join('..', 'models', 'scaler.pkl')
scaler = joblib.load(ruta_scaler)
print("\nScaler cargado correctamente")

# Escalar los datos de ejemplo usando el mismo scaler del entrenamiento
df_ejemplos_std = pd.DataFrame(
    scaler.transform(df_ejemplos),
    index=df_ejemplos.index,
    columns=df_ejemplos.columns
)
print("Datos escalados correctamente")
print("\nPrimeras 5 columnas escaladas (sector energía):")
print(df_ejemplos_std.iloc[0, :5])

# Cargar pipeline guardado
ruta_modelo = os.path.join('..', 'models', 'modelo_final.pkl')
pipeline = joblib.load(ruta_modelo)
print("\nPipeline cargado correctamente")

# Ejecutar pipeline con datos escalados
print("\nEjecutando pipeline con datos de ejemplo escalados...")
predicciones = pipeline.predict(df_ejemplos_std)
probabilidades = pipeline.predict_proba(df_ejemplos_std)

# Mostrar resultados
print("\nResultados de predicción:")
print("="*80)

resultados = []
for i, sector in enumerate(ejemplos.keys()):
    pred = predicciones[i]
    prob_subida = probabilidades[i, 1]
    prob_bajada = probabilidades[i, 0]
    
    resultados.append({
        'Sector': sector,
        'Predicción': 'Subida' if pred == 1 else 'Bajada',
        'P(Subida)': f"{prob_subida:.2%}",
        'P(Bajada)': f"{prob_bajada:.2%}"
    })
    
    print(f"\n{sector.upper()}:")
    print(f"  Predicción: {'SUBIDA' if pred == 1 else 'BAJADA'}")
    print(f"  P(Subida): {prob_subida:.2%}")
    print(f"  P(Bajada): {prob_bajada:.2%}")

# Crear tabla de resultados
df_resultados = pd.DataFrame(resultados)
print("\nTabla de resultados:")
print(df_resultados.to_string(index=False))


# CELDA 3 — Métricas finales del modelo en producción

Recalculamos las métricas finales del modelo sobre el conjunto de prueba
y las comparamos contra la línea base.

# =============================================================================
# CELDA 3 — Métricas finales del modelo en producción
# =============================================================================

# Importar módulo de evaluación y forzar recarga
import importlib
import evaluation as ev
importlib.reload(ev)
print("Módulo evaluation recargado")

import joblib
import os
from sklearn.metrics import roc_curve

# Cargar datos de prueba
ruta_datos = os.path.join('..', 'data', 'processed', 'dataset_modelamiento.csv')
df = pd.read_csv(ruta_datos, index_col=0)

# Preparar datos
activo_objetivo = 'BRENT'
target_col = f'target_{activo_objetivo}'

# Excluir columnas de target y sectores
columnas_excluir = [col for col in df.columns if col.startswith('target_')] + \
                   [col for col in df.columns if col.endswith('_sector')]
X = df.drop(columnas_excluir, axis=1)
y = df[target_col]

# Dividir datos (usar misma semilla que en entrenamiento)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Cargar scaler del entrenamiento y aplicarlo a X_test
ruta_scaler = os.path.join('..', 'models', 'scaler.pkl')
scaler = joblib.load(ruta_scaler)
X_test_std = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)

# Recuperar pipeline guardado
ruta_modelo = os.path.join('..', 'models', 'modelo_final.pkl')
pipeline = joblib.load(ruta_modelo)

# -------------------------------------------------------------------------
# CALCULAR THRESHOLD ÓPTIMO USANDO ÍNDICE DE YOUDEN
# -------------------------------------------------------------------------
y_proba = pipeline.predict_proba(X_test_std)[:, 1]

# Calcular curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# Calcular Índice de Youden
youden_index = tpr - fpr

# Encontrar threshold óptimo
idx_optimo = youden_index.argmax()
threshold_optimo = thresholds[idx_optimo]

print(f"\nThreshold óptimo (Índice de Youden): {threshold_optimo:.4f}")
print(f"  TPR en óptimo: {tpr[idx_optimo]:.3f}")
print(f"  FPR en óptimo: {fpr[idx_optimo]:.3f}")
print(f"  Youden máximo: {youden_index[idx_optimo]:.3f}")

# Hacer predicciones manuales
y_pred_optimo = (y_proba >= threshold_optimo).astype(int)

print(f"\nPredicciones con threshold óptimo ({threshold_optimo:.4f}):")
print(f"  Subidas predichas: {y_pred_optimo.sum()}")
print(f"  Bajadas predichas: {(1-y_pred_optimo).sum()}")

# -------------------------------------------------------------------------
# CALCULAR MÉTRICAS USANDO PREDICCIONES MANUALES
# -------------------------------------------------------------------------
# Pasar y_pred_optimo como argumento posicional (quinto argumento)
metricas = ev.calcular_metricas_completas(
    pipeline, X_test_std, y_test, 'Modelo Final', y_pred_optimo
)

# Comparar contra línea base
linea_base = 0.50  # Línea base de clasificación aleatoria (0.50)

print("\n" + "="*80)
print("COMPARACIÓN CONTRA LÍNEA BASE")
print("="*80)

print(f"\nLínea base de referencia: {linea_base:.2f}")
print(f"\nMétricas del modelo:")
print(f"  AUC-ROC: {metricas['auc']:.4f} (diferencia: {metricas['auc'] - linea_base:+.4f})")
print(f"  F1-Score: {metricas['f1']:.4f}")
print(f"  Accuracy: {metricas['accuracy']:.4f}")
print(f"  Precision: {metricas['precision']:.4f}")
print(f"  Recall: {metricas['recall']:.4f}")

# Conclusión
print("\nConclusión:")
if metricas['auc'] > linea_base:
    print(f"✓ El modelo supera la línea base en {metricas['auc'] - linea_base:.4f} puntos.")
    print("  El modelo agrega valor predictivo significativo.")
else:
    print(f"⚠ El modelo no supera la línea base.")
    print("  Se recomienda revisar el modelo o las features.")


In [ ]:
# =============================================================================
# CELDA 3 — M�tricas finales del modelo en producci�n
# =============================================================================

# Importar m�dulo de evaluaci�n
import evaluation as ev
import joblib
import os
from sklearn.metrics import roc_curve

# Cargar datos de prueba
ruta_datos = os.path.join('..', 'data', 'processed', 'dataset_modelamiento.csv')
df = pd.read_csv(ruta_datos, index_col=0)

# Preparar datos
activo_objetivo = 'BRENT'
target_col = f'target_{activo_objetivo}'

# Excluir columnas de target y sectores
columnas_excluir = [col for col in df.columns if col.startswith('target_')] + \
                   [col for col in df.columns if col.endswith('_sector')]
X = df.drop(columnas_excluir, axis=1)
y = df[target_col]

# Dividir datos (usar misma semilla que en entrenamiento)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Cargar scaler del entrenamiento y aplicarlo a X_test
ruta_scaler = os.path.join('..', 'models', 'scaler.pkl')
scaler = joblib.load(ruta_scaler)
X_test_std = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)

# Recuperar pipeline guardado
ruta_modelo = os.path.join('..', 'models', 'modelo_final.pkl')
pipeline = joblib.load(ruta_modelo)

# Recalcular m�tricas usando X_test escalado
print("Recalculando m�tricas del modelo en producci�n...")

# -------------------------------------------------------------------------
# CALCULAR THRESHOLD �PTIMO USANDO �NDICE DE YOUDEN
# -------------------------------------------------------------------------
y_proba = pipeline.predict_proba(X_test_std)[:, 1]

# Calcular curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# Calcular �ndice de Youden
youden_index = tpr - fpr

# Encontrar threshold �ptimo
idx_optimo = youden_index.argmax()
threshold_optimo = thresholds[idx_optimo]

print(f"\nThreshold �ptimo (�ndice de Youden): {threshold_optimo:.4f}")
print(f"  TPR en �ptimo: {tpr[idx_optimo]:.3f}")
print(f"  FPR en �ptimo: {fpr[idx_optimo]:.3f}")
print(f"  Youden m�ximo: {youden_index[idx_optimo]:.3f}")

# Hacer predicciones manuales
y_pred_optimo = (y_proba >= threshold_optimo).astype(int)

print(f"\nPredicciones con threshold �ptimo ({threshold_optimo:.4f}):")
print(f"  Subidas predichas: {y_pred_optimo.sum()}")
print(f"  Bajadas predichas: {(1-y_pred_optimo).sum()}")

# Pasar y_pred a calcular_metricas_completas
metricas = ev.calcular_metricas_completas(
    pipeline, X_test_std, y_test, 'Modelo Final', y_pred=y_pred_optimo
)

# Comparar contra l�nea base
linea_base = 0.50  # L�nea base de clasificaci�n aleatoria (0.50)

print("\n" + "="*80)
print("COMPARACI�N CONTRA L�NEA BASE")
print("="*80)

print(f"\nL�nea base de referencia: {linea_base:.2f}")
print(f"\nM�tricas del modelo:")
print(f"  AUC-ROC: {metricas['auc']:.4f} (diferencia: {metricas['auc'] - linea_base:+.4f})")
print(f"  F1-Score: {metricas['f1']:.4f}")
print(f"  Accuracy: {metricas['accuracy']:.4f}")
print(f"  Precision: {metricas['precision']:.4f}")
print(f"  Recall: {metricas['recall']:.4f}")

# Conclusi�n
print("\nConclusi�n:")
if metricas['auc'] > linea_base:
    print(f"✓ El modelo supera la l�nea base en {metricas['auc'] - linea_base:.4f} puntos.")
    print("  El modelo agrega valor predictivo significativo.")
else:
    print(f"⚠ El modelo no supera la l�nea base.")
    print("  Se recomienda revisar el modelo o las features.")


# CELDA 4 — Lanzamiento de la app Streamlit desde el notebook

Lanzamos la aplicación Streamlit desde el notebook y documentamos
los pasos para que cualquier usuario pueda reproducir el despliegue.

In [ ]:
# =============================================================================
# CELDA 4 — Lanzamiento de la app Streamlit desde el notebook
# =============================================================================

import subprocess
import time

# Verificar si Streamlit está instalado
try:
    import streamlit
    print("Streamlit ya está instalado.")
except ImportError:
    print("Instalando Streamlit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit"])

# Lanzar app Streamlit
print("\nLanzando aplicación Streamlit...")
print("La app se abrirá en el navegador en: http://localhost:8501")
print("\nPara detener la app, presione Ctrl+C en la terminal.")

# Cambiar al directorio de la app
os.chdir(os.path.join('..', 'app'))

# Lanzar Streamlit en segundo plano
proceso = subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print(f"\nProceso Streamlit iniciado con PID: {proceso.pid}")
print("Esperando 5 segundos para que la app se inicie...")
time.sleep(5)

# Verificar si el proceso está corriendo
if proceso.poll() is None:
    print("✓ App Streamlit corriendo correctamente.")
    print("\nPara acceder a la app, abra su navegador en: http://localhost:8501")
else:
    print("⚠ Error al iniciar la app Streamlit.")
    print("\nSalida de error:")
    print(proceso.stderr.read().decode())

# Documentación para reproducir el despliegue
print("\n" + "="*80)
print("PASOS PARA REPRODUCIR EL DESPLIEGUE DESDE CERO")
print("="*80)

print("\n1. Clonar el repositorio:")
print("   git clone <url_del_repositorio>")
print("   cd proyecto_maduro_mercados")

print("\n2. Crear entorno virtual:")
print("   python -m venv venv")
print("   source venv/bin/activate  # En Windows: venv\Scripts\activate")

print("\n3. Instalar dependencias:")
print("   pip install -r requirements.txt")

print("\n4. Ejecutar notebooks en orden:")
print("   jupyter notebook notebooks/01_preparacion_datos.ipynb")
print("   jupyter notebook notebooks/02_modelos_predictivos.ipynb")
print("   jupyter notebook notebooks/03_despliegue.ipynb")

print("\n5. Lanzar la app Streamlit:")
print("   cd app")
print("   streamlit run streamlit_app.py")

print("\n6. Acceder a la app:")
print("   Abrir navegador en: http://localhost:8501")

# CELDA 5 — Documentación de la interfaz de la app

Descripción detallada de cada sección de la app Streamlit
e instrucciones de uso paso a paso para el usuario final.

## Descripción de la Interfaz de la App Streamlit

### Sección 1: Encabezado
- **Título**: "Predictor de Retorno Anormal Post-Evento Geopolítico"
- **Subtítulo**: "Basado en el evento: Captura de Nicolás Maduro (3 ene 2026)"
- **Información del proyecto**: Nombres de las autoras y descripción del proyecto

### Sección 2: Panel de Entrada (Sidebar)
El panel lateral permite al usuario ingresar los parámetros del activo:

1. **Sector**: Selector desplegable con opciones:
   - energía
   - índice
   - divisa
   - metal
   - volatilidad

2. **Volatilidad 20d**: Slider (0.005 a 0.080, paso 0.001)
   - Representa la volatilidad histórica del activo en los últimos 20 días

3. **Momentum 5d**: Slider (-0.15 a +0.15, paso 0.01)
   - Representa el retorno acumulado en los últimos 5 días

4. **Nivel VIX**: Slider (10 a 80, paso 1)
   - Representa el nivel actual del índice de volatilidad VIX

5. **Correlación con Brent**: Slider (-1.0 a 1.0, paso 0.05)
   - Representa la correlación del activo con el petróleo Brent

6. **CAR Pre-evento**: Slider (-0.20 a +0.20, paso 0.01)
   - Representa el retorno anormal acumulado antes del evento

### Sección 3: Predicción
Botón "Predecir comportamiento del activo" que ejecuta el modelo:

- **Resultado principal**:
  - Si P(subida) > 0.5: "RETORNO ANORMAL POSITIVO (SUBIDA)" en verde
  - Si P(subida) ≤ 0.5: "RETORNO ANORMAL NEGATIVO (BAJADA)" en rojo

- **Gráfico de barras**: Muestra P(Subida) vs P(Bajada)

### Sección 4: Visualización del Clustering
- Muestra el gráfico PCA 2D de clusters de activos
- Texto explicativo del cluster más relevante según el sector seleccionado

### Sección 5: Métricas del Modelo en Producción
Tres columnas con métricas clave:
- **AUC-ROC**: Área bajo la curva ROC
- **F1-Score**: Balance entre precisión y recall
- **Accuracy**: Porcentaje de predicciones correctas

Cada métrica incluye una flecha de comparación vs línea base (0.60)

### Sección 6: Acerca del Proyecto
Expansor con información completa:
- Descripción del proyecto
- Metodología CRISP-DM
- Datos utilizados y fuentes
- Autoras: Laura Laguado y Sofía Navales
- Enlace al repositorio GitHub

---

## Instrucciones de Uso Paso a Paso

### Paso 1: Seleccionar el Sector
En el panel lateral, seleccione el sector del activo que desea analizar:
- **Energía**: Para petróleo (Brent, WTI) y acciones de petroleras (Exxon, Chevron)
- **Índice**: Para índices bursátiles (S&P 500, COLCAP, BOVESPA)
- **Divisa**: Para pares de divisas (USD/COP)
- **Metal**: Para metales preciosos (Oro, Cobre)
- **Volatilidad**: Para índices de volatilidad (VIX)

### Paso 2: Ajustar los Parámetros
Utilice los sliders para ajustar los valores de las características del activo:

1. **Volatilidad 20d**: Ajuste según la volatilidad histórica reciente del activo
   - Valores bajos (< 0.02): Baja volatilidad
   - Valores medios (0.02-0.04): Volatilidad normal
   - Valores altos (> 0.04): Alta volatilidad

2. **Momentum 5d**: Ajuste según la tendencia reciente del activo
   - Valores negativos: Tendencia bajista
   - Cero: Sin tendencia clara
   - Valores positivos: Tendencia alcista

3. **Nivel VIX**: Ajuste según el nivel actual del índice de volatilidad
   - Valores bajos (< 20): Mercado tranquilo
   - Valores medios (20-30): Volatilidad normal
   - Valores altos (> 30): Mercado estresado

4. **Correlación con Brent**: Ajuste según la relación con el petróleo
   - Valores cercanos a 1: Alta correlación positiva
   - Valores cercanos a 0: Sin correlación
   - Valores cercanos a -1: Alta correlación negativa

5. **CAR Pre-evento**: Ajuste según el comportamiento antes del evento
   - Valores negativos: Retorno anormal negativo pre-evento
   - Cero: Sin retorno anormal pre-evento
   - Valores positivos: Retorno anormal positivo pre-evento

### Paso 3: Ejecutar la Predicción
Haga clic en el botón "Predecir comportamiento del activo" para obtener:

1. **Resultado principal**: Indica si el activo generará un retorno anormal
   positivo (subida) o negativo (bajada) ante un evento geopolítico similar

2. **Probabilidades**: Muestra la probabilidad de subida y bajada

3. **Gráfico de barras**: Visualización de las probabilidades

### Paso 4: Interpretar los Resultados

**Si el resultado es SUBIDA (verde)**:
- El modelo predice que el activo generará un retorno anormal positivo
- La probabilidad de subida es mayor al 50%
- Ejemplo: "P(Subida) = 65%" → El modelo es 65% confiable en predecir subida

**Si el resultado es BAJADA (rojo)**:
- El modelo predice que el activo generará un retorno anormal negativo
- La probabilidad de bajada es mayor al 50%
- Ejemplo: "P(Bajada) = 70%" → El modelo es 70% confiable en predecir bajada

### Paso 5: Revisar el Clustering
Observe el gráfico de clustering para entender:
- Qué grupo de activos se comporta de manera similar
- Cómo se relaciona su activo con otros del mismo sector
- Patrones de comportamiento ante eventos geopolíticos

### Paso 6: Evaluar la Confianza del Modelo
Revise las métricas en la sección "Métricas del Modelo en Producción":

- **AUC-ROC > 0.70**: El modelo tiene buena capacidad de discriminación
- **F1-Score > 0.65**: Buen balance entre precisión y recall
- **Accuracy > 0.70**: Buena tasa de predicciones correctas

Si las métricas son cercanas o superiores a la línea base (0.60), el modelo
tiene valor predictivo significativo.

---

## Consejos para el Usuario

1. **Experimente con diferentes parámetros**: Pruebe combinaciones variadas
   para entender cómo cada variable afecta la predicción

2. **Compare sectores**: Ejecute predicciones para diferentes sectores
   y compare los resultados

3. **Considere el contexto**: Las predicciones son basadas en datos históricos
   y eventos similares al de la captura de Maduro

4. **Use con precaución**: El modelo es una herramienta de apoyo, no una
   garantía de comportamiento futuro

5. **Revise la documentación**: Para más detalles sobre la metodología,
   consulte los notebooks en la carpeta `notebooks/`